In [ ]:
from torch.fx.passes import splitter_base
!pip install langchain langchain-groq langchain-community dotenv langchain-text-splitters langchain-huggingface sentence_transformers pypdf bs4 unstructured unstructured[pdf] langchain_ollama faiss-cpu

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

pdfs = DirectoryLoader("./data", glob="*.pdf").load()

len(pdfs)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer, chunk_size=1250, chunk_overlap=150
)

In [ ]:
chunks = splitter.split_documents(pdfs)

len(chunks)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="bge-m3:567m")

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

In [ ]:
from langchain_ollama.llms import OllamaLLM

retriever = vector_store.as_retriever()

model = OllamaLLM(model="gemma3:4b")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente os conteúdo fornecido. Seja breve na resposta \n\nContexto:\n{context}"),
        ("human", "{query}")
    ]
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

In [ ]:
test = "Como fazer um seguro viagem?"

# model.invoke(test)

In [ ]:
results = retriever.invoke(test)
context = "\n\n".join(result.page_content for result in results)

#chain.invoke({"query": test, "context": context})

In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": RunnablePassthrough() | retriever,
        "query": RunnablePassthrough()
    } | prompt | model | StrOutputParser()
)

In [ ]:
# rag_chain.invoke(test)

In [ ]:
query_model = OllamaLLM(model="gemma3:1b")

rewriter_prompt_template = """
Gere consulta de pesquisa para o banco de dados de vetores (Vector DB) a partir de uma pergunta do usuário,
permitindo uma resposta mais precisa por meio da busca semantica.
Basta retornar a consulta revisada do Vector DB, entre aspas.

Pergunta do usuário: {user_question}

Consulta revisada do Vector DB:
"""

from langchain_core.prompts import PromptTemplate

rewriter_prompt = PromptTemplate.from_template(rewriter_prompt_template)

rewriter_chain = rewriter_prompt | query_model | StrOutputParser()

rewriter_chain.invoke(test)

In [ ]:
rewriter_rag_chain = (
    {
        "context": RunnablePassthrough() | rewriter_chain | retriever,
        "query": RunnablePassthrough()
    } | prompt | model | StrOutputParser()
)

In [ ]:
rewriter_rag_chain.invoke(test)

In [ ]:
multi_query_prompt_template = """Você é um assistente de modelo de linguagem de IA. Sua tarefa é gerar cinco
versões diferentes da pergunta do usuário para recuperar documentos relevantes de um banco de dados vetorial.
Ao gerar múltiplas perspectivas sobre a pergunta do usuário, seu objetivo é ajudar
o usuário a superar algumas das limitações da busca por similaridade baseada em distância.
Forneça estas perguntas alternativas separadas por quebras de linha. A Resposta deve conter apenas os 5 resultados separados por ','
Pergunta original: {question}"""

from langchain_core.output_parsers import CommaSeparatedListOutputParser

multi_query_prompt = PromptTemplate.from_template(multi_query_prompt_template)

multi_query_chain = multi_query_prompt | query_model | CommaSeparatedListOutputParser()

In [ ]:
multi_query_chain.invoke(test)

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_retriever = MultiQueryRetriever(
    retriever=retriever, llm_chain=multi_query_chain
)

multi_rag_chain = (
    {
        "context": RunnablePassthrough() | multi_retriever,
        "query": RunnablePassthrough()
    }
    | prompt | model | StrOutputParser()
)

In [ ]:
multi_rag_chain.invoke(test)

In [ ]:
hyde_prompt_template = """
Responda a pergunta da melhor forma possivel em no minimo 2 paragrafos.
Pergunta: {user_question}
Responda:
"""

hyde_prompt = PromptTemplate.from_template(rewriter_prompt_template)

hyde_chain = hyde_prompt | query_model | StrOutputParser()

hyde_chain.invoke(test)

In [ ]:
hyde_rag_chain = (
    {
        "context": RunnablePassthrough() | hyde_chain | retriever,
        "query": RunnablePassthrough()
    }
    | prompt | model | StrOutputParser()
)

In [ ]:
hyde_rag_chain.invoke(test)

In [ ]:
from langchain_classic.evaluation.qa import QAEvalChain

eval_chain = QAEvalChain.from_llm(model)

def evaluate(question_answers, generation):
    correct = 0
    evaluation = eval_chain(question_answers, generation)
    for i, e in enumerate(question_answers):
        correct = correct + (1 if evaluation[i]["results"].split("\n")[-1].split(":")[-1].strip() == "CORRECT" else 0)
    return correct/len(question_answers)

In [ ]:
from langchain_classic.evaluation.qa import QAGenerateChain

qa_chain = QAGenerateChain.from_llm(model)

question_answers_eval = qa_chain.apply_and_parse([{"doc": p.page_content} for p in chunks])

In [ ]:
import json

with open("qa_pairs.json") as file:
    pairs = json.load(file)

In [ ]:
results_eval = []

for qa in question_answers_eval:
    results_eval.append({"result": rewriter_chain.invoke(qa["question"])})

evaluate(qa, results_eval)